# LLMeter with OpenAI Response API on Amazon Bedrock

This notebook demonstrates how to use LLMeter to measure latency of LLMs hosted on [Amazon Bedrock](https://aws.amazon.com/bedrock/) via the [OpenAI Response API](https://platform.openai.com/docs/api-reference/responses).

The Response API is OpenAI's newer interface that offers structured outputs, instructions-based system prompts, and improved multi-turn conversation handling. Bedrock exposes this API through its Mantle endpoint for supported models.

We cover the basics of invoking models through LLMeter's `ResponseEndpoint` and `ResponseStreamEndpoint`, running request batches, and analyzing the results.

## Setting Up

First, ensure your environment has LLMeter installed with the plotting and OpenAI extras.

In [ ]:
%pip install "llmeter[plotting]<1" openai aws-bedrock-token-generator

In [ ]:
from aws_bedrock_token_generator import provide_token
from upath import UPath

from llmeter.endpoints import ResponseEndpoint, ResponseStreamEndpoint
from llmeter.runner import Runner

This notebook assumes you're running in an environment with [configured AWS API credentials](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/credentials.html) and that the configured identity has permissions to invoke Bedrock models.

The Response API on Bedrock uses the **Mantle endpoint** (`bedrock-mantle.<region>.api.aws`) rather than the standard `bedrock-runtime` endpoint. Authentication is handled via a temporary token generated by `aws-bedrock-token-generator`.

### Key differences from Chat Completions API

| | Chat Completions | Response API |
|---|---|---|
| Input field | `messages` | `input` (string or message array) |
| Token limit | `max_tokens` | `max_output_tokens` |
| System prompt | System message in `messages` | `instructions` parameter |
| Response text | `choices[0].message.content` | `output_text` helper |
| Endpoint URL | `bedrock-runtime.<region>.amazonaws.com/openai/v1` | `bedrock-mantle.<region>.api.aws/v1` |
| Model ID format | With version suffix (e.g. `openai.gpt-oss-120b-1:0`) | Without version suffix (e.g. `openai.gpt-oss-120b`) |

In [ ]:
import os

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
MODEL_ID = "openai.gpt-oss-120b"  # Response API uses model ID without version suffix
BASE_URL = f"https://bedrock-mantle.{AWS_REGION}.api.aws/v1"

# Generate a temporary token for Bedrock authentication
token = provide_token(region=AWS_REGION)

print(f"Region:   {AWS_REGION}")
print(f"Model:    {MODEL_ID}")
print(f"Endpoint: {BASE_URL}")

## Creating payloads

LLMeter's `ResponseEndpoint` provides a `create_payload` helper that builds a correctly formatted request for the Response API.

In [ ]:
sample_payload = ResponseEndpoint.create_payload(
    "Create a list of 3 pop songs.",
    max_output_tokens=512,
    instructions="You are an expert in pop and indie music.",
)
sample_payload

## Basic inference (non-streaming)

We create a `ResponseEndpoint` and verify it's working correctly.

In [ ]:
endpoint = ResponseEndpoint(
    model_id=MODEL_ID,
    api_key=token,
    base_url=BASE_URL,
)

In [ ]:
response = endpoint.invoke(payload=sample_payload)
print(response)

You should see `time_to_last_token` (overall response time) captured in the response. Token counts (`num_tokens_input`, `num_tokens_output`) depend on whether the provider returns usage data. `time_to_first_token` is null here because it's only measured in streaming mode.

## Running a test batch

A single data point doesn't give us much confidence about typical endpoint performance. We can use the `Runner` class to send multiple requests and calculate statistics.

In [ ]:
endpoint_test = Runner(
    endpoint,
    output_path=f"outputs/{endpoint.model_id}-response",
)
results = await endpoint_test.run(payload=sample_payload, n_requests=3, clients=3)

In [ ]:
print(results)

In [ ]:
results.responses

## Streaming responses and Time-to-First-Token

The Response API also supports streaming, which reduces user-perceived latency by sending the response in chunks. LLMeter's `ResponseStreamEndpoint` captures both `time_to_first_token` (TTFT) and `time_to_last_token` (TTLT).

In [ ]:
stream_endpoint = ResponseStreamEndpoint(
    model_id=MODEL_ID,
    api_key=token,
    base_url=BASE_URL,
)

In [ ]:
response = stream_endpoint.invoke(payload=sample_payload)
print(response)

Now we can see `time_to_first_token` is populated alongside `time_to_last_token`. Let's run a larger batch to get meaningful statistics.

In [ ]:
stream_test = Runner(
    stream_endpoint,
    output_path=f"outputs/{stream_endpoint.model_id}-response-stream",
)
results_stream = await stream_test.run(
    payload=sample_payload, clients=20, n_requests=5
)

In [ ]:
print(results_stream)

In [ ]:
results_stream.responses

## Analysis of the results

Let's visualize the streaming results using LLMeter's plotting utilities.

In [ ]:
from llmeter.plotting import (
    boxplot_by_dimension,
    histogram_by_dimension,
    scatter_histogram_2d,
)
import plotly.graph_objects as go

First, let's look at the distribution of input and output token counts across our test requests.

In [ ]:
scatter_histogram_2d(
    results_stream,
    x_dimension="num_tokens_input",
    y_dimension="num_tokens_output",
    n_bins_x=20,
    n_bins_y=20,
)

With a constant input prompt, the distribution of time-to-first-token is particularly interesting.

In [ ]:
fig = go.Figure(layout=dict(title="Time to first token - histogram"))

fig.add_trace(
    histogram_by_dimension(
        results_stream, dimension="time_to_first_token", xbins=dict(size=0.02)
    ),
)

In [ ]:
fig = go.Figure(layout=dict(title="Time to first token - boxplot"))

fig.add_trace(
    boxplot_by_dimension(
        results_stream, dimension="time_to_first_token"
    ),
)

We can also check the relationship between time per output token (TPOT) and the number of output tokens.

In [ ]:
scatter_histogram_2d(
    results_stream,
    x_dimension="num_tokens_output",
    y_dimension="time_per_output_token",
    n_bins_x=20,
    n_bins_y=20,
)

## Saving and loading results

If `output_path` was set, results are saved to files automatically.

In [ ]:
print(f"Contents of {results_stream.output_path}\n----")
for sub in UPath(results_stream.output_path).iterdir():
    print(sub)